In [2]:
!pip install requests beautifulsoup4 tqdm urllib3 sentence-transformers faiss-cpu pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 12.6 MB/s eta 0:00:00


In [3]:
!pip install spacy networkx pyvis streamlit-option-menu -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import sqlite3

base_dir = "/content/drive/MyDrive/PolicyNav/documents"
db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

# Create folder
os.makedirs(base_dir, exist_ok=True)

# Create database + table
conn = sqlite3.connect(db_path)

c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS policies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    url TEXT,
    local_path TEXT
)
""")

conn.commit()
conn.close()

print("Database initialized successfully!")
print("PDF folder:", base_dir)
print("Database:", db_path)

Database initialized successfully!
PDF folder: /content/drive/MyDrive/PolicyNav/documents
Database: /content/drive/MyDrive/PolicyNav/policies_meta.db


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm
import urllib3
import time
import os
import sqlite3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Paths
base_dir = "/content/drive/MyDrive/PolicyNav/documents"
db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

os.makedirs(base_dir, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# Known Official Policy PDFs
# -----------------------------
# -----------------------------
# Known Official Policy PDFs
# -----------------------------
direct_pdfs = [
    # -- Your Original Scrapes --
    ("NEP","https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Final_English_0.pdf"),
    ("PMJDY","https://www.pmjdy.gov.in/files/E-Documents/PMJDY_Brochure_ENG.pdf"),
    ("PMAY_Urban","https://pmay-urban.gov.in/uploads/guidelines/Operational-Guidelines-of-PMAY-U.pdf"),
    ("PMAY_Gramin","https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf"),
    ("PMFBY","https://pmfby.gov.in/pdf/Revamped%20Operational%20Guidelines_17th%20August%202020.pdf"),
    ("PMKISAN","https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf"),
    ("JalJeevanMission","https://jaljeevanmission.gov.in/sites/default/files/guideline/JJM_Operational_Guidelines.pdf"),
    ("SHC","https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf"),
    ("SwachhBharat","https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf"),
    ("PMKVY","https://www.msde.gov.in/static/uploads/2024/02/PMKVY-4.0-Guidelines_final-copy.pdf"),
    ("DigitalIndia","https://www.meity.gov.in/static/uploads/2024/03/Brochure.pdf"),
    ("SmartCities","https://mohua.gov.in/dataSmartCities/uploads/resource/resourceDoc/Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf"),

    # -- NEW ADDITIONS: Healthcare & Nutrition --
    ("AyushmanBharat_PMJAY", "https://pmjay.gov.in/sites/default/files/2018-07/Guidelines_on_Process_of_Empanelment_of_Hospitals_0.pdf"),
    ("Poshan_Abhiyaan", "https://icds-wcd.nic.in/nnm/NNM-Web-Contents/UPPER-MENU/AboutNNM/PM_Overarching_Scheme_Guidelines.pdf"),

    # -- NEW ADDITIONS: Employment & Rural --
    ("MGNREGA", "https://nrega.nic.in/netnrega/writereaddata/Circulars/MGNREGA_Operational_Guidelines_2013_eng.pdf"),
    ("AMRUT_Mission", "https://mohua.gov.in/upload/uploadfiles/files/AMRUTGuidelines.pdf"),

    # -- NEW ADDITIONS: Infrastructure & Energy --
    ("PM_Ujjwala_Yojana", "https://www.pmuy.gov.in/documents/Ujjwala-Brochure.pdf"),
    ("Saubhagya_Electricity", "https://saubhagya.gov.in/documents/Guidelines_Saubhagya.pdf"),

    # -- NEW ADDITIONS: Finance & Pensions --
    ("Atal_Pension_Yojana", "https://npscra.nsdl.co.in/nsdl/scheme-details/APY_Scheme_Details.pdf"),
    ("StandUp_India", "https://www.standupmitra.in/Content/StandupIndia/SUISchemeGuidelines.pdf")
]


# -----------------------------
# Policy Landing Pages
# -----------------------------
policy_pages = {
    # -- Your Original Scrapes --
    "PMJDY":"https://pmjdy.gov.in/",
    "PMAY_Urban":"https://pmay-urban.gov.in/",
    "JalJeevanMission":"https://jaljeevanmission.gov.in/",
    "DigitalIndia":"https://digitalindia.gov.in/",
    "SkillIndia":"https://www.msde.gov.in/",

    # -- NEW ADDITIONS --
    "AyushmanBharat": "https://pmjay.gov.in/",
    "StartupIndia": "https://www.startupindia.gov.in/",
    "MakeInIndia": "https://www.makeinindia.com/",
    "WomenAndChildDev": "https://wcd.nic.in/",           # Pulls Beti Bachao Beti Padhao, etc.
    "RuralDevelopment": "https://rural.nic.in/",         # Pulls Rural infrastructure schemes
    "MudraYojana": "https://www.mudra.org.in/",
    "PM_SVANidhi": "https://pmsvanidhi.mohua.gov.in/",   # Street vendor loans
    "MinistryOfHealth": "https://main.mohfw.gov.in/"
}


# -----------------------------
# Find PDFs from websites
# -----------------------------
def find_pdfs(url):

    try:
        r = requests.get(url,headers=HEADERS,timeout=20,verify=False)

        soup = BeautifulSoup(r.text,"html.parser")

        pdfs = []

        for link in soup.find_all("a",href=True):

            if ".pdf" in link["href"].lower():

                pdfs.append(urljoin(url,link["href"]))

        return pdfs

    except:

        return []


# -----------------------------
# Check already downloaded
# -----------------------------
def already_downloaded(url):

    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute("SELECT id FROM policies WHERE url=?",(url,))
    data = c.fetchone()

    conn.close()

    return data is not None


# -----------------------------
# Download PDF (CORRECTED)
# -----------------------------
def download_pdf(url,scheme):
    filename = url.split("/")[-1].split("?")[0]
    save_name = f"{scheme}_{filename}"
    path = os.path.join(base_dir,save_name)

    # If it exists, skip instantly!
    if os.path.exists(path) or already_downloaded(url):
        return False # Returns False meaning "Did not download"

    try:
        r = requests.get(url,headers=HEADERS,stream=True,timeout=30,verify=False)
        if "application/pdf" not in r.headers.get("Content-Type","").lower() and not r.content.startswith(b"%PDF"):
            return False

        with open(path,"wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

        conn = sqlite3.connect(db_path)
        c = conn.cursor()
        c.execute("INSERT INTO policies (title,url,local_path) VALUES (?,?,?)", (scheme,url,path))
        conn.commit()
        conn.close()

        print("Downloaded:",save_name)
        return True # Returns True meaning "Successfully downloaded"

    except:
        print("Failed:",url)
        return False

# -----------------------------
# Combine sources & Download (CORRECTED)
# -----------------------------
pdf_mapping = []

# Add direct policy PDFs
for scheme,url in direct_pdfs:
    pdf_mapping.append((url,scheme))

print("Scanning policy websites...")
for scheme,url in policy_pages.items():
    pdfs = find_pdfs(url)
    print(f"{scheme}: {len(pdfs)} PDFs found")
    for pdf in pdfs:
        pdf_mapping.append((pdf,scheme))

print("\nTotal PDFs found:",len(pdf_mapping))
print("Checking Drive for new files...\n")

for pdf_url,scheme in pdf_mapping:
    # Only trigger the 1-second pause if a brand new file was actually downloaded
    was_downloaded = download_pdf(pdf_url,scheme)
    if was_downloaded:
        time.sleep(1)


# -----------------------------
# Final Count
# -----------------------------
conn = sqlite3.connect(db_path)
c = conn.cursor()

c.execute("SELECT COUNT(*) FROM policies")
count = c.fetchone()[0]

conn.close()

print("\nDownload complete!")
print("Total PDFs stored in database:",count)

Scanning policy websites...
PMJDY: 7 PDFs found
PMAY_Urban: 41 PDFs found
JalJeevanMission: 35 PDFs found
DigitalIndia: 4 PDFs found
SkillIndia: 0 PDFs found
AyushmanBharat: 0 PDFs found
StartupIndia: 3 PDFs found
MakeInIndia: 0 PDFs found
WomenAndChildDev: 0 PDFs found
RuralDevelopment: 0 PDFs found
MudraYojana: 0 PDFs found
PM_SVANidhi: 0 PDFs found
MinistryOfHealth: 0 PDFs found

Total PDFs found: 110
Checking Drive for new files...

Failed: https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf
Failed: https://icds-wcd.nic.in/nnm/NNM-Web-Contents/UPPER-MENU/AboutNNM/PM_Overarching_Scheme_Guidelines.pdf
Failed: https://npscra.nsdl.co.in/nsdl/scheme-details/APY_Scheme_Details.pdf
Failed: https://www.standupmitra.in/Content/StandupIndia/SUISchemeGuidelines.pdf

Download complete!
Total PDFs stored in database: 350


In [ ]:
from pypdf import PdfReader
import os

base_dir = "/content/drive/MyDrive/PolicyNav/documents"

documents = []

for file in os.listdir(base_dir):
    if file.endswith(".pdf"):
        path = os.path.join(base_dir, file)
        try:
            reader = PdfReader(path)
            text = ""
            for page in reader.pages:
                if page.extract_text():
                    text += page.extract_text()

            documents.append({
                "file": file,
                "text": text
            })
        except:
            print("Error reading:", file)

print("Total PDFs processed:", len(documents))

Error reading: education.gov.in_ugc_act.pdf


Total PDFs processed: 353


In [ ]:
def split_text(text, chunk_size=500):

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

all_chunks = []
for doc in documents:
    chunks = split_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({
            "source": doc["file"],
            "text": chunk
        })
print("Total chunks created:", len(all_chunks))

NameError: name 'documents' is not defined

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/185 [00:00<?, ?it/s]

Embedding shape: (5900, 384)


In [ ]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print("Vectors stored:", index.ntotal)

Vectors stored: 5900


In [ ]:
faiss.write_index(index, "/content/drive/MyDrive/PolicyNav/policy_vector_db.index")
import pickle
with open("/content/drive/MyDrive/PolicyNav/chunks.pkl","wb") as f:
    pickle.dump(all_chunks,f)
print("Vector database saved!")

Vector database saved!


In [ ]:
import pickle
import faiss
index = faiss.read_index("/content/drive/MyDrive/PolicyNav/policy_vector_db.index")
with open("/content/drive/MyDrive/PolicyNav/chunks.pkl","rb") as f:
    chunks = pickle.load(f)
def search_policy(query, top_k=5):
    query_vector = model.encode([query])
    distances, indices = index.search(query_vector, top_k)
    results = []
    for i in indices[0]:
        results.append(chunks[i]["text"])
    return results

In [ ]:
query = "What are benefits of PM Kisan scheme?"
results = search_policy(query)
for r in results:
    print(r)
    print("\n---\n")

RU-16-03-0002-231121/FAQ Pradhan Mantri Kisan Samman Nidhi (PM- KISAN) (Ministry of Agriculture and Farmers Welfare) Frequently Asked Questions (FAQs) November 23, 2021 1. What is Pradhan Mantri Kisan Samman Nidhi? Pradhan Mantri Kisan Samman Nidhi (PM-KISAN) is a new Central Sector Scheme to provide income support to all landholding farmers' families in the country to supplement their financial needs for procuring various inputs related to agriculture and allied activities as well as domestic needs. Under the Scheme, the entire financial liability towards transfer of benefit to targeted beneficiaries will be borne by the Government of India. 2. When was the scheme launched? The PM -KISAN Scheme was launched by the Prime Minister Narendra Modi on February 24, 2019. 3. From which date the Scheme has come into effect? The scheme takes effect from December 01,2018. 4. What are the benefits of the Scheme? Under the PM-KISAN scheme, all landholding farmers' families shall be provided the fi

In [ ]:
def generate_answer(query):

    context = "\n".join(search_policy(query))
    prompt = f"""
    Answer the question using the policy context.
    Context:
    {context}
    Question:
    {query}
    """
    response = llm(prompt)

    return response

In [6]:
pip install transformers sentencepiece torch accelerate -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANSLATOR_ID = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ID)

translator_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATOR_ID,
    device_map="auto"
)

print("Translator model loaded successfully")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translator model loaded successfully


In [ ]:
LANG_CODES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Kannada": "kan_Knda",
    "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

In [ ]:
def translate_fast(text, source_lang, target_lang):

    if source_lang == target_lang:
        return text

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    translator_tokenizer.src_lang = src_code

    inputs = translator_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(translator_model.device)

    tgt_token_id = translator_tokenizer.convert_tokens_to_ids(tgt_code)

    outputs = translator_model.generate(
        **inputs,
        forced_bos_token_id=tgt_token_id,
        max_length=512
    )

    return translator_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM loaded")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

LLM loaded


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def multilingual_rag(question, user_lang="English"):

    # Translate user query → English
    english_query = translate_fast(question, user_lang, "English")

    print("Original Question:", question)
    print("Translated Query:", english_query)

    # Vector search
    docs = search_policy(english_query)

    context = "\n\n".join(docs)

    prompt = f"""
Answer the question using the policy context.

Context:
{context}

Question:
{english_query}
"""

    # Generate English answer
    english_answer = llm(prompt)[0]["generated_text"]

    # Translate answer back
    final_answer = translate_fast(english_answer, "English", user_lang)

    return final_answer

# -------------------------------------------------------------
#             SUMMARIZATION MODEL & FUNCTION
# -------------------------------------------------------------
if 'sum_tokenizer' not in globals():
    print("Loading dedicated Summarization Model...")
    sum_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    sum_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")

def summarize_policy(text, user_lang="English"):
    """Summarizes policy text and translates it to the target language."""
    print("⏳ Generating summary using AI...")

    prompt = f"Summarize the following text into 3 key bullet points:\n{text[:2000]}"

    # Generate English Summary
    inputs = sum_tokenizer(prompt, return_tensors="pt", truncation=True).to(sum_model.device)
    outputs = sum_model.generate(**inputs, max_new_tokens=200)
    english_summary = sum_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Translate it
    return translate_fast(english_summary, "English", user_lang)

Loading dedicated Summarization Model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
!pip install duckduckgo_search -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 45.1 MB/s eta 0:00:00


In [7]:
import spacy
import networkx as nx
from pyvis.network import Network
import pandas as pd
from duckduckgo_search import DDGS
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, display

# Load NLP for Knowledge Graph
nlp = spacy.load("en_core_web_sm")

# --- PART 1: GLOBAL RESEARCH (Web Search Logic) ---
def global_web_research(query):
    """Searches the live web for global context/updates."""
    print(f"🔍 Searching the web for: {query}...")
    results = []
    with DDGS() as ddgs:
        # Get top 3 web results
        search_results = [r for r in ddgs.text(query, max_results=3)]
        for res in search_results:
            results.append({
                "title": res['title'],
                "body": res['body'],
                "href": res['href']
            })
    return results

# --- PART 2: KNOWLEDGE GRAPH (Entity Extraction) ---
def generate_knowledge_graph(chunks):
    """Builds a relationship map from your PDF data."""
    net = Network(height="500px", width="100%", bgcolor="#2b2b2b", font_color="white", notebook=True)

    # Process a subset of chunks to prevent clutter
    subset = chunks[:15]

    for chunk in subset:
        filename = chunk["source"]
        doc = nlp(chunk["text"])

        # Central Node (The Document)
        net.add_node(filename, label=filename, color="#00ff1e", size=25)

        # Entity Nodes (The content inside the document)
        for ent in doc.ents:
            if ent.label_ in ["ORG", "GPE", "DATE"]: # Organizations, Locations, Dates
                net.add_node(ent.text, label=ent.text, color="#4da3ff", size=15)
                net.add_edge(filename, ent.text, title="mentions")

    graph_html = "policy_graph.html"
    net.show(graph_html)
    return graph_html

# --- PART 3: UNIFIED INTERFACE (The Navigation) ---
#@title 🚀 Milestone 3 Dashboard { run: "auto" }
Select_Module = "Summarization" #@param ["Knowledge Graph", "Global Research", "Summarization"]
User_Query = "How does PM Kisan compare to global farm subsidies?" #@param {type:"string"}
Target_Language = "Hindi" #@param ["English", "Hindi", "Tamil", "Telugu", "Kannada", "Marathi", "Bengali"]

if Select_Module == "Knowledge Graph":
    print("✨ Extracting Entities from your PDFs...")
    if 'all_chunks' in globals():
        graph_file = generate_knowledge_graph(all_chunks)
        display(HTML(graph_file))
    else:
        print("❌ Error: 'all_chunks' not found. Please run your PDF processing code first!")

elif Select_Module == "Global Research":
    print("🌐 Performing Live Web Research...")
    research_data = global_web_research(User_Query)
    for i, item in enumerate(research_data):
        print(f"\n[{i+1}] {item['title']}")
        print(f"Source: {item['href']}")
        print(f"Summary: {item['body'][:200]}...")
        print("-" * 50)

# -------------------------------------------------------------
#                   THE SUMMARIZATION UI
# -------------------------------------------------------------
elif Select_Module == "Summarization":
    print("📝 Generating Document Summary...")
    if 'all_chunks' in globals() and len(all_chunks) > 0:
        # Grab the first chunk of text from the PDFs
        sample_text = all_chunks[0]['text']
        source_doc = all_chunks[0]['source']

        print(f"\n📄 Extracted from: {source_doc}")
        print("-" * 50)

        summary_result = summarize_policy(sample_text, user_lang=Target_Language)
        print(f"\n✨ Final Summary ({Target_Language}):\n{summary_result}")
    else:
        print("❌ Error: 'all_chunks' not found. Please run the PDF processing code first!")

📝 Generating Document Summary...
❌ Error: 'all_chunks' not found. Please run the PDF processing code first!


In [ ]:
# query = "पीएम किसान योजनेचे फायदे काय आहेत?"

# answer = multilingual_rag(
#     question=query,
#     user_lang="Marathi"
# )

# print("\nFinal Answer:\n", answer)

In [ ]:
# --- MILESTONE 3: TEST & VERIFICATION CELL ---
from IPython.display import HTML, display
import os

def run_final_check():
    print("🚀 Starting Final System Check...\n")

    # 1. Check if Data Chunks exist
    if 'all_chunks' in globals() and len(all_chunks) > 0:
        print(f"✅ Success: Found {len(all_chunks)} text chunks for processing.")
    else:
        print("❌ Error: 'all_chunks' is missing. Run the PDF processing cells first.")
        return

    # 2. Test Knowledge Graph Generation
    print("📊 Testing Knowledge Graph Generation...")
    try:
        # We call your existing function
        graph_path = generate_knowledge_graph(all_chunks[:10]) # Test with first 10 for speed
        if os.path.exists(graph_path):
            print(f"✅ Success: Knowledge Graph created at '{graph_path}'.")
            display(HTML(filename=graph_path))
        else:
            print("❌ Error: Graph file was not generated.")
    except Exception as e:
        print(f"❌ Error in KG Logic: {e}")

    # 3. Test Global Research (Web Search)
    print("\n🌐 Testing Global Research (Live Web Search)...")
    try:
        test_query = "Latest updates on PM Kisan scheme 2026"
        research_results = global_web_research(test_query)
        if research_results and len(research_results) > 0:
            print(f"✅ Success: Found {len(research_results)} live web results.")
            print(f"Top Result: {research_results[0]['title']}")
        else:
            print("❌ Error: Web search returned no results.")
    except Exception as e:
        print(f"❌ Error in Research Logic: {e}")

    print("\n🏁 Check Complete! If you see the graph above and success messages, you are ready to submit.")
# Execute the test
run_final_check()

🚀 Starting Final System Check...

❌ Error: 'all_chunks' is missing. Run the PDF processing cells first.


In [8]:
!pip install pyngrok streamlit transformers sentencepiece torch accelerate spacy networkx pyvis pandas beautifulsoup4 PyPDF2 textstat pyjwt bcrypt wikipedia -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.5 MB/s eta 0:00:00


In [9]:
%%writefile styles.py
CSS = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');

    /* ===================== GLOBAL ===================== */
    .stApp, p, h1, h2, h3, h4, h5, h6, span, div, input, button {
        font-family: 'Inter', sans-serif;
    }

    span.material-symbols-rounded,
    i.material-icons,
    [data-testid="collapsedControl"] span,
    [data-testid="baseButton-header"] span {
        font-family: 'Material Symbols Rounded', 'Material Icons', sans-serif !important;
    }

    /* Gemini-style Dark Backgrounds */
    .stApp { background: #131314 !important; }
    .main  { background: #131314 !important; }

    #MainMenu { visibility: hidden; }
    footer    { visibility: hidden; }
    header { background: transparent !important; }

    /* ===================== SIDEBAR ===================== */
    section[data-testid="stSidebar"] {
        background-color: #1e1f20 !important;
        border-right: 1px solid #444746 !important;
    }

    section[data-testid="stSidebar"] .stScrollableContainer {
        padding: 1.5rem 0.8rem !important;
    }

    button[data-testid="collapsedControl"] {
        color: #c4c7c5 !important;
        background-color: transparent !important;
        border: none !important;
        transition: background 0.2s ease !important;
        padding: 0.5rem !important;
    }

    button[data-testid="collapsedControl"]:hover {
        background-color: #282a2c !important;
        border-radius: 8px !important;
    }

    .sidebar-spacer { min-height: 45vh; }

    .menu-label {
        color: #8e918f;
        font-size: 0.75rem;
        font-weight: 600;
        padding: 0.5rem 0.8rem;
        margin-top: 0.5rem;
        letter-spacing: 0.05em;
        text-transform: uppercase;
    }

    /* Sidebar buttons */
    section[data-testid="stSidebar"] .stButton > button {
        background: transparent !important;
        color: #c4c7c5 !important;
        font-size: 0.9rem !important;
        padding: 0.65rem 0.8rem !important;
        font-weight: 400 !important;
        border-radius: 8px !important;
        border: none !important;
        box-shadow: none !important;
        margin: 2px 0 !important;
        text-align: left !important;
        display: flex !important;
        justify-content: flex-start !important;
        transition: background 0.15s ease, color 0.15s ease !important;
    }

    section[data-testid="stSidebar"] .stButton > button:hover {
        background: #282a2c !important;
        color: #e3e3e3 !important;
    }

    /* Selected state */
    section[data-testid="stSidebar"] .stButton > button[kind="primary"] {
        background: #004a77 !important;
        color: #c2e7ff !important;
        font-weight: 500 !important;
    }

    /* Clean Sidebar User Profile block */
    .sidebar-bottom-profile {
        display: flex;
        align-items: center;
        padding: 12px 14px;
        border-radius: 8px;
        cursor: pointer;
        transition: background 0.2s;
        border-top: 1px solid #444746;
        margin-top: 0.5rem;
    }

    .sidebar-bottom-profile:hover {
        background: #282a2c;
    }

    .profile-name {
        color: #e3e3e3;
        font-size: 0.95rem;
        font-weight: 500;
        white-space: nowrap;
        text-overflow: ellipsis;
        overflow: hidden;
        display: flex;
        align-items: center;
    }

    .main .block-container {
        padding: 2.5rem 3rem !important;
        max-width: 100% !important;
    }

    /* ===================== LOGO ===================== */
    .logo-container {
        text-align: center;
        margin-bottom: 1.5rem;
        padding-bottom: 1rem;
    }

    .logo-icon {
        display: inline-flex;
        padding: 0.7rem;
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        margin-bottom: 0.8rem;
    }

    .logo-icon svg {
        width: 2rem;
        height: 2rem;
        color: #a8c7fa;
    }

    .logo-text {
        font-size: 1.4rem;
        font-weight: 600;
        color: #e3e3e3;
        letter-spacing: -0.02em;
        margin: 0;
    }

    .logo-subtext {
        color: #8e918f;
        font-size: 0.75rem;
        letter-spacing: 0.05em;
        text-transform: uppercase;
        margin: 0;
    }

    /* ===================== PAGE TITLE ===================== */
    .page-title {
        font-size: 1.8rem;
        font-weight: 500;
        color: #e3e3e3;
        margin-bottom: 0.3rem;
        letter-spacing: -0.03em;
    }

    .page-subtitle {
        color: #8e918f;
        font-size: 0.9rem;
        margin-bottom: 2rem;
    }

    /* ===================== INPUTS ===================== */
    .stTextInput > label,
    .stSelectbox > label {
        color: #c4c7c5 !important;
        font-size: 0.8rem !important;
        font-weight: 500 !important;
        margin-bottom: 0.4rem !important;
    }

    .stTextInput > div > div > input {
        background: #1e1f20 !important;
        border: 1px solid #444746 !important;
        border-radius: 8px !important;
        padding: 0.7rem 1rem !important;
        color: #e3e3e3 !important;
        font-size: 0.9rem !important;
        transition: all 0.2s ease !important;
    }

    .stTextInput > div > div > input:focus {
        border-color: #a8c7fa !important;
        box-shadow: none !important;
        background: #282a2c !important;
    }

    /* ===================== BUTTONS ===================== */
    .main .stButton > button {
        background: #a8c7fa !important;
        color: #062e6f !important;
        border: none !important;
        border-radius: 8px !important;
        padding: 0.65rem 1rem !important;
        font-size: 0.9rem !important;
        font-weight: 500 !important;
        width: 100% !important;
        margin: 0.3rem 0 !important;
        transition: opacity 0.2s ease !important;
        box-shadow: none !important;
    }

    .main .stButton > button:hover {
        opacity: 0.9 !important;
        transform: translateY(-1px) !important;
    }

    .main .stButton > button[kind="secondary"] {
        background: #1e1f20 !important;
        color: #a8c7fa !important;
        border: 1px solid #444746 !important;
    }

    .main .stButton > button[kind="secondary"]:hover {
        background: #282a2c !important;
        border-color: #a8c7fa !important;
        transform: none !important;
    }

    /* ===================== ALERTS ===================== */
    .stAlert {
        background: #1e1f20 !important;
        border: 1px solid #444746 !important;
        border-radius: 8px !important;
        color: #e3e3e3 !important;
    }

    /* ===================== ADMIN BADGE ===================== */
    .admin-badge {
        background: #004a77;
        color: #c2e7ff;
        padding: 4px 12px;
        border-radius: 20px;
        font-size: 0.65rem;
        font-weight: 600;
        letter-spacing: 0.08em;
        text-transform: uppercase;
        display: inline-block;
    }

    /* ===================== DASHBOARD STAT CARDS ===================== */
    .stat-card {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.8rem 1.5rem;
        text-align: center;
        transition: all 0.2s ease;
        position: relative;
        overflow: hidden;
    }

    .stat-number {
        font-size: 2.2rem;
        font-weight: 500;
        color: #e3e3e3;
        margin: 0;
        letter-spacing: -0.04em;
        font-family: 'DM Mono', monospace !important;
    }

    .stat-label {
        color: #8e918f;
        font-size: 0.75rem;
        margin: 0.3rem 0 0 0;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        font-weight: 500;
    }

    /* ===================== READABILITY METRIC CARDS ===================== */
    .metric-card {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.4rem;
        position: relative;
    }

    .metric-title {
        color: #8e918f;
        font-size: 0.78rem;
        font-weight: 600;
        text-transform: uppercase;
        margin-bottom: 0.3rem;
        letter-spacing: 0.05em;
    }

    .metric-value {
        font-size: 2rem;
        font-weight: 500;
        color: #e3e3e3;
        font-family: 'DM Mono', monospace !important;
        margin: 0.5rem 0 0.8rem 0;
    }

    .metric-bar-track {
        height: 4px;
        background: #282a2c;
        border-radius: 4px;
        overflow: hidden;
        margin-bottom: 0.5rem;
    }

    .metric-bar-fill {
        height: 100%;
        border-radius: 4px;
        transition: width 1s ease;
    }

    .metric-interpretation {
        font-size: 0.75rem;
        color: #c4c7c5;
        margin: 0;
    }

    /* Level Banner */
    .level-banner {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.8rem 2rem;
        display: flex;
        align-items: center;
        gap: 1.5rem;
        margin-bottom: 1.5rem;
    }

    .level-icon {
        width: 56px; height: 56px;
        border-radius: 12px;
        display: flex; align-items: center; justify-content: center;
        font-size: 1.6rem;
    }

    .level-title { font-size: 1.3rem; font-weight: 500; color: #e3e3e3; margin: 0 0 0.2rem 0; }
    .level-desc { font-size: 0.82rem; color: #8e918f; margin: 0; }
    .level-grade { margin-left: auto; text-align: center; }
    .level-grade-num { font-size: 2.4rem; font-weight: 500; font-family: 'DM Mono', monospace !important; color: #e3e3e3; margin: 0; line-height: 1; }
    .level-grade-label { font-size: 0.72rem; color: #8e918f; text-transform: uppercase; letter-spacing: 0.05em; }

    /* Text Stat Pills */
    .stat-row { display: flex; gap: 0.8rem; margin-top: 1rem; flex-wrap: wrap; }
    .stat-pill { background: #1e1f20; border: 1px solid #444746; border-radius: 10px; padding: 0.8rem 1.2rem; flex: 1; min-width: 100px; text-align: center; }
    .stat-pill-value { font-size: 1.4rem; font-weight: 500; color: #e3e3e3; font-family: 'DM Mono', monospace !important; display: block; }
    .stat-pill-label { font-size: 0.7rem; color: #8e918f; text-transform: uppercase; font-weight: 500; letter-spacing: 0.05em;}

    /* Tooltips */
    .tooltip-wrap { position: relative; display: inline-block; cursor: help; }
    .tooltip-icon { width: 14px; height: 14px; background: #282a2c; border-radius: 50%; display: inline-flex; align-items: center; justify-content: center; font-size: 0.6rem; color: #8e918f; }
    .tooltip-box { visibility: hidden; opacity: 0; background: #282a2c; border: 1px solid #444746; border-radius: 8px; padding: 0.8rem 1rem; width: 220px; position: absolute; bottom: 130%; left: 50%; transform: translateX(-50%); z-index: 9999; box-shadow: 0 4px 12px rgba(0,0,0,0.3); pointer-events: none; }
    .tooltip-wrap:hover .tooltip-box { visibility: visible; opacity: 1; }
    .tooltip-box p { color: #c4c7c5; font-size: 0.78rem; margin: 0; line-height: 1.5; text-transform: none; font-weight: 400; }
    .tooltip-box strong { color: #e3e3e3; display: block; margin-bottom: 0.3rem; font-size: 0.8rem; text-transform: none; }

    /* Tabs */
    .stTabs [data-baseweb="tab-list"] { background: transparent !important; border-bottom: 1px solid #444746 !important; border-radius: 0 !important; padding: 0 !important; }
    .stTabs [data-baseweb="tab"] { background: transparent !important; color: #8e918f !important; border-radius: 0 !important; padding: 0.8rem 1rem !important; border: none !important; }
    .stTabs [aria-selected="true"] { background: transparent !important; color: #a8c7fa !important; border-bottom: 2px solid #a8c7fa !important; }

    /* File Uploader */
    .stFileUploader > div { background: #1e1f20 !important; border: 1px dashed #444746 !important; border-radius: 8px !important; }

    /* Text Area */
    .stTextArea textarea { background: #1e1f20 !important; border: 1px solid #444746 !important; border-radius: 8px !important; color: #e3e3e3 !important; }
    .stTextArea textarea:focus { border-color: #a8c7fa !important; box-shadow: none !important; }
</style>
"""
print("styles.py created successfully!")

Writing styles.py


In [10]:
%%writefile templates.py
class Templates:

    @staticmethod
    def logo():
        return """
        <div class="logo-container">
            <div class="logo-icon">
                <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                    <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
                </svg>
            </div>
            <div class="logo-text">PolicyNav</div>
            <div class="logo-subtext">Secure Access Management</div>
        </div>
        """

    @staticmethod
    def divider():
        return '<div class="divider"></div>'

    @staticmethod
    def info_box(text):
        return f'<div class="info-box">{text}</div>'

    @staticmethod
    def welcome_message(username):
        return f"""
        <div class="welcome-text">
            Welcome, <span class="username-highlight">{username}</span>!
        </div>
        """
print("templates.py created successfully!")

Writing templates.py


In [11]:
%%writefile templates.py
class Templates:
    @staticmethod
    def logo():
        return """
        <div class="logo-container" style="text-align: center; margin-bottom: 1.5rem; padding-bottom: 1rem;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/9/95/Infosys_logo.svg" style="width: 130px; margin-bottom: 0.8rem;">
            <p class="logo-subtext" style="color: #8e918f; font-size: 0.75rem; letter-spacing: 0.05em; text-transform: uppercase; margin: 0;">PolicyNav | Secure Access Management</p>
        </div>
        """

Overwriting templates.py


In [12]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }
print("readability.py created successfully!")

Writing readability.py


In [13]:
!pip install pandas plotly matplotlib wordcloud

In [22]:
%%writefile app.py
import streamlit as st
import sqlite3
import jwt
import datetime
import hashlib
import re
import time
import secrets
import smtplib
import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os
from styles import CSS
from templates import Templates
import readability
import PyPDF2
import streamlit.components.v1 as components

# --- DATA SCIENCE & ANALYTICS IMPORTS ---
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# --- AI IMPORTS ---
import pickle
import faiss
import spacy
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pyvis.network import Network
import wikipedia

# ================= LOAD SECRETS =================
EMAIL_ADDRESS = os.environ.get('EMAIL_ID', 'your_email@gmail.com')
EMAIL_PASSWORD = os.environ.get('EMAIL_APP_PASSWORD', 'your_app_password')
SECRET_KEY = os.environ.get('JWT_SECRET', 'fallback-secret-key-change-me')
ADMIN_EMAIL    = os.environ.get('ADMIN_EMAIL', 'admin@policynav.com')
ADMIN_PASSWORD = os.environ.get('ADMIN_PASSWORD', 'Admin@123')

# ================= PAGE CONFIG =================
st.set_page_config(page_title="PolicyNav", layout="wide", initial_sidebar_state="expanded")
st.markdown(CSS, unsafe_allow_html=True)

# Professional, Vibrant SaaS Sidebar CSS & Uploader Fix
st.markdown("""
<style>
[data-testid="stSidebar"] { background-color: #0b0f19; }
[data-testid="stSidebar"] .menu-label { color: #60a5fa; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 2px; margin: 35px 0 15px 15px; font-weight: 700; border-bottom: 1px solid #1e293b; padding-bottom: 5px; }
[data-testid="stSidebar"] .stButton button { width: 90%; margin: 5px auto; border-radius: 10px; border: 1px solid transparent; text-align: left; justify-content: flex-start; padding: 14px 20px; font-weight: 500; color: #94a3b8; background-color: transparent; transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1); }
[data-testid="stSidebar"] .stButton button p { font-size: 1.1rem; }
[data-testid="stSidebar"] .stButton button:hover { background-color: #1e293b; color: #ffffff; transform: translateX(8px); border-left: 5px solid #3b82f6; box-shadow: 0 4px 15px rgba(0,0,0,0.3); }
[data-testid="stSidebar"] .stButton button[kind="primary"] { background: linear-gradient(90deg, #1e3a8a 0%, #1e40af 100%); color: #ffffff !important; font-weight: 600; border-left: 5px solid #60a5fa; box-shadow: 0 4px 20px rgba(37, 99, 235, 0.4); }

/* 👇 Absolute Override for Streamlit Uploader Text 👇 */
[data-testid="stFileUploadDropzone"] small,
[data-testid="stFileUploader"] small,
[data-testid="stFileUploadDropzone"] div[data-testid="stMarkdownContainer"] p,
[data-testid="stFileUploadDropzone"] > div > small,
section[data-testid="stFileUploadDropzone"] small {
    display: none !important;
    font-size: 0px !important;
    color: transparent !important;
}
</style>
""", unsafe_allow_html=True)

# ================= AI MODEL CACHING =================
@st.cache_resource
def load_data_and_models():
    index_path = "/content/drive/MyDrive/PolicyNav/policy_vector_db.index"
    chunks_path = "/content/drive/MyDrive/PolicyNav/chunks.pkl"
    if os.path.exists(index_path) and os.path.exists(chunks_path):
        index = faiss.read_index(index_path)
        with open(chunks_path, "rb") as f: chunks = pickle.load(f)
    else:
        index, chunks = None, [{"source": "No Data", "text": "Run PDF processor first."}]
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    t_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
    t_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", device_map="auto")
    s_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    s_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")
    nlp = spacy.load("en_core_web_sm")
    return index, chunks, embed_model, t_tokenizer, t_model, s_tokenizer, s_model, nlp

# ================= AI HELPER FUNCTIONS =================
LANG_CODES = {"English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml", "Telugu": "tel_Telu", "Kannada": "kan_Knda", "Marathi": "mar_Deva", "Bengali": "ben_Beng", "Malayalam": "mal_Mlym", "Gujarati": "guj_Gujr", "Urdu": "urd_Arab"}

def translate_fast(text, source_lang, target_lang, t_tokenizer, t_model):
    if source_lang == target_lang: return text
    t_tokenizer.src_lang = LANG_CODES.get(source_lang, "eng_Latn")
    inputs = t_tokenizer(text, return_tensors="pt", truncation=True).to(t_model.device)
    tgt_id = t_tokenizer.convert_tokens_to_ids(LANG_CODES.get(target_lang, "eng_Latn"))
    outputs = t_model.generate(**inputs, forced_bos_token_id=tgt_id, max_length=512)
    return t_tokenizer.decode(outputs[0], skip_special_tokens=True)

def search_policy(query, index, chunks, embed_model, top_k=5):
    if not index: return ["No database found."]
    query_vector = embed_model.encode([query])
    distances, indices = index.search(query_vector, top_k)
    return [chunks[i]["text"] for i in indices[0]]

def generate_knowledge_graph(chunks_subset, nlp):
    net = Network(height="450px", width="100%", bgcolor="#131314", font_color="#e3e3e3", notebook=False)
    for chunk in chunks_subset:
        filename = chunk["source"]
        doc = nlp(chunk["text"])
        net.add_node(filename, label=filename, color="#a8c7fa", size=20)
        for ent in doc.ents:
            if ent.label_ in ["ORG", "GPE", "DATE"]:
                net.add_node(ent.text, label=ent.text, color="#c4c7c5", size=12)
                net.add_edge(filename, ent.text, title="mentions")
    net.save_graph("policy_graph.html")

def global_web_research(query):
    results = []
    try:
        search_results = wikipedia.search(query, results=3)
        for title in search_results:
            summary = wikipedia.summary(title, sentences=2, auto_suggest=False)
            url = wikipedia.page(title, auto_suggest=False).url
            results.append({"title": title, "body": summary, "href": url})
    except:
        results.append({"title": "Notice", "body": "Could not fetch results.", "href": ""})
    return results

# ================= AUTH & DB CONFIG =================
ALGORITHM, TOKEN_EXPIRE_MINUTES, MAX_LOGIN_ATTEMPTS, LOCKOUT_TIME, OTP_EXPIRY_MINUTES = "HS256", 30, 3, 300, 10
DB_PATH = "/content/drive/MyDrive/PolicyNav/users.db"
conn   = sqlite3.connect(DB_PATH, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT NOT NULL, email TEXT UNIQUE NOT NULL, password TEXT NOT NULL, security_question TEXT NOT NULL, security_answer TEXT NOT NULL, created_at TEXT, is_blocked INTEGER DEFAULT 0)""")
try: cursor.execute("ALTER TABLE users ADD COLUMN is_blocked INTEGER DEFAULT 0"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE users ADD COLUMN avatar TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'"); conn.commit()
except: pass

cursor.execute("""CREATE TABLE IF NOT EXISTS password_history (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT NOT NULL, password TEXT NOT NULL, set_at TEXT, FOREIGN KEY(email) REFERENCES users(email))""")
cursor.execute("""CREATE TABLE IF NOT EXISTS login_attempts (email TEXT PRIMARY KEY, attempts INTEGER DEFAULT 0, last_attempt REAL)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS otp_requests (email TEXT PRIMARY KEY, otp TEXT, expires_at REAL)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS user_activity (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, action_type TEXT, details TEXT, timestamp TEXT)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, rating INTEGER, comments TEXT, timestamp TEXT)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS tool_feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, tool_name TEXT, prompt TEXT, response TEXT, rating TEXT, comments TEXT, timestamp TEXT)""")

try: cursor.execute("ALTER TABLE tool_feedback ADD COLUMN comments TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE user_activity ADD COLUMN prompt TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE user_activity ADD COLUMN response TEXT"); conn.commit()
except: pass
conn.commit()

# ================= UTILS & VALIDS =================
def _get_timestamp(): return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
def hash_password(password): return hashlib.sha256(password.encode()).hexdigest()
def create_token(email, username, role="user"): return jwt.encode({"sub": email, "username": username, "role": role, "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=TOKEN_EXPIRE_MINUTES)}, SECRET_KEY, algorithm=ALGORITHM)
def valid_email(email): return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email)

# ================= DB LOGIC =================
def log_activity(email, action_type, details, prompt="", response=""):
    cursor.execute("INSERT INTO user_activity (email, action_type, details, prompt, response, timestamp) VALUES (?, ?, ?, ?, ?, ?)", (email, action_type, details, prompt, response, _get_timestamp())); conn.commit()

def get_user_history(email):
    cursor.execute("SELECT action_type, details, prompt, response, timestamp FROM user_activity WHERE email = ? AND action_type NOT IN ('System', 'Security', 'Profile') ORDER BY id DESC", (email,))
    return cursor.fetchall()

def submit_feedback(email, rating, comments):
    cursor.execute("INSERT INTO feedback (email, rating, comments, timestamp) VALUES (?, ?, ?, ?)", (email, rating, comments, _get_timestamp())); conn.commit()

def submit_tool_feedback(email, tool_name, prompt, response, rating, comments):
    cursor.execute("INSERT INTO tool_feedback (email, tool_name, prompt, response, rating, comments, timestamp) VALUES (?, ?, ?, ?, ?, ?, ?)", (email, tool_name, prompt, response, rating, comments, _get_timestamp()))
    conn.commit()

def get_all_tool_feedback():
    cursor.execute("SELECT email, tool_name, prompt, response, rating, comments, timestamp FROM tool_feedback ORDER BY id DESC")
    return cursor.fetchall()

def get_user_avatar(email):
    cursor.execute("SELECT avatar FROM users WHERE email = ?", (email,))
    res = cursor.fetchone()
    return res[0] if res else None

def update_avatar(email, base64_string):
    cursor.execute("UPDATE users SET avatar = ? WHERE email = ?", (base64_string, email))
    conn.commit()

def update_user_email_db(old_email, new_email):
    cursor.execute("UPDATE users SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE password_history SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE login_attempts SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE otp_requests SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE user_activity SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE feedback SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE tool_feedback SET email = ? WHERE email = ?", (new_email, old_email))
    conn.commit()

def get_all_feedback(): cursor.execute("SELECT email, rating, comments, timestamp FROM feedback ORDER BY id DESC"); return cursor.fetchall()
def get_login_attempts(email): cursor.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,)); data = cursor.fetchone(); return data if data else (0, 0)
def increment_login_attempts(email): attempts, _ = get_login_attempts(email); cursor.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)", (email, attempts + 1, time.time())); conn.commit()
def reset_login_attempts(email): cursor.execute("DELETE FROM login_attempts WHERE email = ?", (email,)); conn.commit()
def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= MAX_LOGIN_ATTEMPTS:
        if time.time() - last_attempt < LOCKOUT_TIME: return True, LOCKOUT_TIME - (time.time() - last_attempt)
        else: reset_login_attempts(email)
    return False, 0
def register_user(username, email, password, security_question, security_answer):
    try:
        now, hashed_pass, hashed_answer = _get_timestamp(), hash_password(password), hash_password(security_answer.strip())
        cursor.execute("INSERT INTO users (username, email, password, security_question, security_answer, created_at) VALUES (?, ?, ?, ?, ?, ?)", (username, email, hashed_pass, security_question, hashed_answer, now))
        cursor.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed_pass, now)); conn.commit(); return True
    except sqlite3.IntegrityError: return False

def authenticate_user(email, password):
    is_limited, _ = is_rate_limited(email)
    if is_limited: return False, "locked"
    cursor.execute("SELECT username, password, is_blocked, role FROM users WHERE email = ?", (email,)); user = cursor.fetchone()
    if user:
        if user[2] == 1: return False, "blocked"
        if user[1] == hash_password(password): reset_login_attempts(email); return True, (user[0], user[3])
    increment_login_attempts(email); return False, None

def authenticate_admin(email, password): return email == ADMIN_EMAIL and password == ADMIN_PASSWORD
def check_user_exists(email): cursor.execute("SELECT 1 FROM users WHERE email = ?", (email,)); return cursor.fetchone() is not None
def get_user_details(email): cursor.execute("SELECT username, security_question, security_answer FROM users WHERE email = ?", (email,)); return cursor.fetchone()
def check_password_reused(email, new_password):
    cursor.execute("SELECT password FROM password_history WHERE email = ? ORDER BY id DESC LIMIT 5", (email,)); history = cursor.fetchall(); hashed_new = hash_password(new_password)
    for (stored_hash,) in history:
        if stored_hash == hashed_new: return True
    return False
def update_password(email, new_password):
    hashed, now = hash_password(new_password), _get_timestamp()
    cursor.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email)); cursor.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now)); conn.commit(); reset_login_attempts(email)
def verify_security_answer(email, answer): cursor.execute("SELECT security_answer FROM users WHERE email = ?", (email,)); stored_answer = cursor.fetchone(); return stored_answer and stored_answer[0] == hash_password(answer.strip())

def get_all_users(): cursor.execute("SELECT id, username, email, created_at, is_blocked, role FROM users ORDER BY id DESC"); return cursor.fetchall()
def toggle_block_user(email, current_status):
    new_status = 0 if current_status == 1 else 1
    cursor.execute("UPDATE users SET is_blocked = ? WHERE email = ?", (new_status, email)); conn.commit()
def toggle_role_user(email, current_role):
    new_role = 'admin' if current_role == 'user' else 'user'
    cursor.execute("UPDATE users SET role = ? WHERE email = ?", (new_role, email)); conn.commit()
def delete_user(email):
    cursor.execute("DELETE FROM users WHERE email = ?", (email,)); cursor.execute("DELETE FROM password_history WHERE email = ?", (email,))
    cursor.execute("DELETE FROM login_attempts WHERE email = ?", (email,)); cursor.execute("DELETE FROM otp_requests WHERE email = ?", (email,))
    cursor.execute("DELETE FROM user_activity WHERE email = ?", (email,)); cursor.execute("DELETE FROM feedback WHERE email = ?", (email,))
    cursor.execute("DELETE FROM tool_feedback WHERE email = ?", (email,)); conn.commit()
def get_user_stats():
    cursor.execute("SELECT COUNT(*) FROM users"); total = cursor.fetchone()[0]; cursor.execute("SELECT COUNT(*) FROM users WHERE is_blocked = 1"); blocked = cursor.fetchone()[0]
    return total, blocked, total - blocked

def generate_otp(): return f"{secrets.randbelow(1000000):06d}"
def save_otp(email, otp): expires_at = time.time() + (OTP_EXPIRY_MINUTES * 60); cursor.execute("INSERT OR REPLACE INTO otp_requests (email, otp, expires_at) VALUES (?, ?, ?)", (email, otp, expires_at)); conn.commit()
def verify_otp(email, otp):
    cursor.execute("SELECT otp, expires_at FROM otp_requests WHERE email = ?", (email,)); data = cursor.fetchone()
    if data and data[0] == otp and time.time() < data[1]: cursor.execute("DELETE FROM otp_requests WHERE email = ?", (email,)); conn.commit(); return True
    return False
def send_otp_email(to_email, otp):
    try:
        msg = MIMEMultipart(); msg['From'] = f"PolicyNav <{EMAIL_ADDRESS}>"; msg['To'] = to_email; msg['Subject'] = "PolicyNav Password Reset"
        body = f"<html><body><h1 style='color: #e3e3e3; text-align: center;'>{otp}</h1></body></html>"
        msg.attach(MIMEText(body, 'html')); server = smtplib.SMTP('smtp.gmail.com', 587); server.starttls(); server.login(EMAIL_ADDRESS, EMAIL_PASSWORD); server.send_message(msg); server.quit()
        return True, "OTP sent successfully"
    except Exception as e: return False, str(e)

# ================= SESSION =================
_session_defaults = {"page": "login", "token": None, "role": "user", "reset_email": None, "security_question": None, "otp_verified": False, "username": None, "email": None, "menu_option": "Dashboard", "reset_method": None, "otp_sent": False}
for _k, _v in _session_defaults.items():
    if _k not in st.session_state: st.session_state[_k] = _v

def metric_card(label, tooltip_desc, value, bar_pct, bar_color):
    bar_pct_clamped = max(0, min(100, bar_pct))
    return f"""<div class="metric-card"><div class="metric-title">{label}</div><div class="metric-value">{value:.1f}</div><div class="metric-bar-track"><div class="metric-bar-fill" style="width:{bar_pct_clamped}%; background:{bar_color};"></div></div><p class="metric-interpretation">{tooltip_desc}</p></div>"""

# ================= PAGES =================
def dashboard_page(username, chunks):
    now_dt = datetime.datetime.now()
    _, center_col, _ = st.columns([1, 6, 1])
    with center_col:
        st.markdown(f"<div style='margin-bottom: 2rem; margin-top: 2rem; text-align: center;'><p style='color: #a8c7fa; font-size: 0.85rem; margin: 0 0 0.2rem 0; text-transform: uppercase; letter-spacing: 0.05em;'>Welcome back</p><h1 style='margin-bottom: 0.2rem; font-size: 2.5rem; color: #e3e3e3;'>{username}</h1><p style='color: #8e918f; font-size: 0.9rem; margin: 0;'>{now_dt.strftime('%A, %B %d, %Y')}</p></div>", unsafe_allow_html=True)

        st.markdown("<h3 style='text-align: center; color: #e3e3e3; margin-top: 1rem;'>Latest Policy Updates</h3>", unsafe_allow_html=True)
        st.markdown("<p style='color: #8e918f; font-size: 0.9rem; text-align: center; margin-bottom: 1.5rem;'>Hover over the feed to pause scrolling and read.</p>", unsafe_allow_html=True)

        unique_srcs = []
        for c in reversed(chunks):
            if c["source"] not in unique_srcs and c["source"] != "No Data":
                unique_srcs.append(c["source"])
            if len(unique_srcs) >= 5: break

        cards_html = ""
        if not unique_srcs:
            cards_html = "<div class='policy-card'><p class='policy-title'>No Policies Found</p><p class='policy-desc'>Please process PDFs to populate the database.</p></div>"
        else:
            for src in unique_srcs:
                first_chunk = ""
                for c in chunks:
                    if c["source"] == src:
                        first_chunk = c["text"]
                        break
                title = src.replace(".pdf", "").replace("_", " ").title()
                desc = first_chunk[:130] + "..." if len(first_chunk) > 130 else first_chunk
                cards_html += f"<div class='policy-card'><p class='policy-title'>{title}</p><p class='policy-desc'>{desc}</p><span class='policy-tag'>Recently Added to Database</span></div>"

        ticker_css = "<style>.news-ticker-container {height: 350px; overflow: hidden; background-color: #1e1f20; border-radius: 12px; border: 1px solid #444746; position: relative; padding: 10px 20px; box-shadow: 0 10px 20px rgba(0,0,0,0.4); } .news-scroll {display: flex; flex-direction: column; gap: 15px; animation: scroll-vertical 20s linear infinite; } .news-ticker-container:hover .news-scroll {animation-play-state: paused; } .policy-card {background-color: #131314; padding: 18px; border-radius: 8px; border-left: 5px solid #a8c7fa; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom:10px;} .policy-title {color: #a8c7fa; font-size: 1.15rem; font-weight: 600; margin: 0 0 8px 0; } .policy-desc {color: #e3e3e3; font-size: 0.95rem; margin: 0; line-height: 1.5; } .policy-tag {display: inline-block; background-color: #2e3032; color: #81c995; font-size: 0.75rem; padding: 3px 8px; border-radius: 4px; margin-top: 10px; } @keyframes scroll-vertical {0% { transform: translateY(100%); } 100% { transform: translateY(-100%); } } </style>"
        ticker_html = f"<div class='news-ticker-container'><div class='news-scroll'>{cards_html}</div></div>"

        st.markdown(ticker_css + ticker_html, unsafe_allow_html=True)

def signup():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Create Account</h1>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        col1, col2 = st.columns(2)
        with col1:
            username = st.text_input("Username", placeholder="your_username", key="signup_username")
            password = st.text_input("Password", type="password", placeholder="••••••••", key="signup_password")
            security_question = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favorite teacher?"], key="signup_question")
        with col2:
            email = st.text_input("Email", placeholder="you@example.com", key="signup_email")
            confirm = st.text_input("Confirm Password", type="password", placeholder="••••••••", key="signup_confirm")
            security_answer = st.text_input("Security Answer", placeholder="Your answer", key="signup_answer")
        if st.button("Create Account", key="signup_button", use_container_width=True, type="primary"):
            if not all([username, email, password, confirm, security_answer]): st.error("All fields required"); return
            if password != confirm: st.error("Passwords do not match"); return
            if register_user(username, email, password, security_question, security_answer):
                st.success("Account created successfully"); time.sleep(1.5); st.session_state.page = "login"; st.rerun()
            else: st.error("Registration failed or Email exists")
        if st.button("Back to Login", key="back_to_login", use_container_width=True): st.session_state.page = "login"; st.rerun()

def login():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Welcome back</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; margin-bottom: 2rem;'>Sign in to your account</p>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        email = st.text_input("Email", placeholder="you@example.com", key="login_email")
        password = st.text_input("Password", type="password", placeholder="••••••••", key="login_password")
        if st.button("Sign In", key="login_button", use_container_width=True, type="primary"):
            if not email or not password: st.error("Please enter email and pass both")
            else:
                auth_result, status = authenticate_user(email, password)
                if auth_result:
                    username, role = status
                    st.session_state.token = create_token(email, username, role)
                    st.session_state.username = username
                    st.session_state.email = email
                    st.session_state.role = role
                    st.session_state.page = "dashboard"
                    log_activity(email, "System", "User logged in successfully")
                    st.rerun()
                else:
                    if status == "locked": st.error("Account locked. Try again later.")
                    elif status == "blocked": st.error("Account blocked by admin.")
                    else: st.error("Invalid credentials")
        st.markdown("<br>", unsafe_allow_html=True)
        c1, c2, c3 = st.columns(3)
        with c1:
            if st.button("Create Account", use_container_width=True): st.session_state.page = "signup"; st.rerun()
        with c2:
            if st.button("Forgot Password", use_container_width=True): st.session_state.page = "forgot"; st.rerun()
        with c3:
            if st.button("Admin", use_container_width=True): st.session_state.page = "admin_login"; st.rerun()

def admin_login():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align:center;'>Admin Access</h1>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        admin_email = st.text_input("Admin Email", key="admin_email")
        admin_pass = st.text_input("Admin Password", type="password", key="admin_pass")
        if st.button("Access Admin Panel", type="primary", use_container_width=True):
            if authenticate_admin(admin_email, admin_pass):
                st.session_state.token = "admin_token"
                st.session_state.username = "Admin"
                st.session_state.email = admin_email
                st.session_state.role = "admin"
                st.session_state.page = "dashboard"
                st.rerun()
            else: st.error("Invalid admin credentials")
        if st.button("Back to User Login", use_container_width=True): st.session_state.page = "login"; st.rerun()

def forgot_password():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Reset Password</h1>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        if not st.session_state.reset_email:
            email = st.text_input("Email", placeholder="Your registered email", key="forgot_email")
            st.markdown("<p style='text-align:center; margin:1rem 0 0.5rem; font-size:0.85rem;'>Choose your verification method</p>", unsafe_allow_html=True)
            col_otp, col_sec = st.columns(2)
            with col_otp: otp_btn = st.button("Via OTP Email", use_container_width=True)
            with col_sec: sec_btn = st.button("Via Security Q&A", use_container_width=True)
            if otp_btn or sec_btn:
                if not email: st.error("Please enter your email first")
                elif not check_user_exists(email): st.error("Email not found in our system")
                elif otp_btn:
                    otp = generate_otp(); save_otp(email, otp); success, msg = send_otp_email(email, otp)
                    if success: st.session_state.reset_email = email; st.session_state.reset_method = "otp"; st.success("OTP sent to your email."); time.sleep(1); st.rerun()
                    else: st.error(f"Failed to send OTP: {msg}")
                else: st.session_state.reset_email = email; st.session_state.reset_method = "security"; st.rerun()
        elif st.session_state.reset_method == "otp" and not st.session_state.otp_verified:
            otp_input = st.text_input("Enter OTP", placeholder="6-digit code", max_chars=6)
            if st.button("Verify OTP", type="primary", use_container_width=True):
                if verify_otp(st.session_state.reset_email, otp_input): st.session_state.otp_verified = True; st.success("OTP verified."); st.rerun()
                else: st.error("Invalid or expired OTP")
        elif st.session_state.reset_method == "security" and not st.session_state.otp_verified:
            user_details = get_user_details(st.session_state.reset_email)
            if user_details:
                st.info(f"Question: {user_details[1]}")
                answer = st.text_input("Your Answer", placeholder="Enter your answer")
                if st.button("Verify Answer", type="primary", use_container_width=True):
                    if verify_security_answer(st.session_state.reset_email, answer): st.session_state.otp_verified = True; st.success("Answer verified."); time.sleep(0.8); st.rerun()
                    else: st.error("Incorrect security answer")
        elif st.session_state.otp_verified:
            new_password = st.text_input("New Password", type="password", placeholder="••••••••")
            confirm_password = st.text_input("Confirm Password", type="password", placeholder="••••••••")
            if st.button("Reset Password", type="primary", use_container_width=True):
                if new_password == confirm_password and not check_password_reused(st.session_state.reset_email, new_password):
                    update_password(st.session_state.reset_email, new_password); st.success("Password updated successfully."); time.sleep(1.5)
                    st.session_state.page = "login"; st.session_state.reset_email = None; st.session_state.otp_verified = False; st.rerun()
                else: st.error("Passwords mismatch or reused old password")
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("Back", use_container_width=True): st.session_state.page = "login"; st.session_state.reset_email = None; st.session_state.otp_verified = False; st.rerun()

# ================= MAIN ROUTING =================
if st.session_state.token:
    is_admin = st.session_state.role == "admin"
    username = st.session_state.username
    if not is_admin:
        with st.spinner("Initializing Workspace..."): index, chunks, embed_model, t_tokenizer, t_model, s_tokenizer, s_model, nlp = load_data_and_models()

    with st.sidebar:
        st.markdown("<div style='display:flex; align-items:center; gap:15px; margin-bottom:2rem; padding: 0 10px;'><img src='https://upload.wikimedia.org/wikipedia/commons/9/95/Infosys_logo.svg' style='width:75px;'/><span style='color:#ffffff;font-weight:700;font-size:1.3rem;letter-spacing:-0.02em;'>PolicyNav</span></div>", unsafe_allow_html=True)
        if is_admin:
            st.markdown("<div class='menu-label'>ADMINISTRATION</div>", unsafe_allow_html=True)
            if st.button("System Dashboard", use_container_width=True, type="primary" if st.session_state.menu_option == "Dashboard" else "secondary"): st.session_state.menu_option = "Dashboard"; st.rerun()
        else:
            st.markdown("<div class='menu-label'>AI INTELLIGENCE</div>", unsafe_allow_html=True)
            if st.button("User Dashboard", use_container_width=True, type="primary" if st.session_state.menu_option == "Dashboard" else "secondary"): st.session_state.menu_option = "Dashboard"; st.rerun()
            if st.button("AI Policy Assistant", use_container_width=True, type="primary" if st.session_state.menu_option == "Chat" else "secondary"): st.session_state.menu_option = "Chat"; st.rerun()
            if st.button("AI Policy Summarizer", use_container_width=True, type="primary" if st.session_state.menu_option == "Summarization" else "secondary"): st.session_state.menu_option = "Summarization"; st.rerun()
            if st.button("Entity Knowledge Graph", use_container_width=True, type="primary" if st.session_state.menu_option == "Graph" else "secondary"): st.session_state.menu_option = "Graph"; st.rerun()
            if st.button("Global Web Search", use_container_width=True, type="primary" if st.session_state.menu_option == "Web" else "secondary"): st.session_state.menu_option = "Web"; st.rerun()
            if st.button("Readability Analyzer", use_container_width=True, type="primary" if st.session_state.menu_option == "Readability" else "secondary"): st.session_state.menu_option = "Readability"; st.rerun()
            if st.button("Language Translator", use_container_width=True, type="primary" if st.session_state.menu_option == "Translation" else "secondary"): st.session_state.menu_option = "Translation"; st.rerun()

            st.markdown("<div class='menu-label'>PERSONAL PORTAL</div>", unsafe_allow_html=True)
            if st.button("My Activity History", use_container_width=True, type="primary" if st.session_state.menu_option == "History" else "secondary"): st.session_state.menu_option = "History"; st.rerun()
            if st.button("Submit Feedback", use_container_width=True, type="primary" if st.session_state.menu_option == "Feedback" else "secondary"): st.session_state.menu_option = "Feedback"; st.rerun()
            if st.button("User Profile", use_container_width=True, type="primary" if st.session_state.menu_option == "Profile" else "secondary"): st.session_state.menu_option = "Profile"; st.rerun()

        st.markdown("<div style='flex-grow: 1;'></div>", unsafe_allow_html=True)
        if st.button("Log out", key="logout_btn", use_container_width=True):
            log_activity(st.session_state.email, "System", "User logged out")
            for key in list(st.session_state.keys()): del st.session_state[key]
            st.rerun()

        user_avatar = None
        if not is_admin: user_avatar = get_user_avatar(st.session_state.email)

        if user_avatar: avatar_html = f"<img src='data:image/png;base64,{user_avatar}' style='width:28px; height:28px; border-radius:50%; margin-right:10px; object-fit:cover; border: 1px solid #444746;'/>"
        else: avatar_html = f"<div style='width:24px; height:24px; border-radius:50%; background:#444746; display:flex; align-items:center; justify-content:center; margin-right:10px; font-size:0.7rem; color:#e3e3e3;'>{username[0].upper()}</div>"
        st.markdown(f"<div style='padding:20px; border-top:1px solid #1e293b; color:#94a3b8; font-size:0.9rem;'>Logged in as: <br><div style='display:flex; align-items:center; margin-top:8px;'>{avatar_html}<b style='color:#ffffff;'>{username}</b></div></div>", unsafe_allow_html=True)

    # --- MAIN CONTENT AREA ROUTING ---
    if is_admin:
        if st.session_state.menu_option == "Dashboard":
            st.markdown("<h2>Admin Command Center</h2>", unsafe_allow_html=True)

            tab1, tab2, tab3, tab4 = st.tabs(["User Management", "System Analytics", "Feedback Analysis", "Data Exports"])

            with tab1:
                total, blocked, active = get_user_stats()
                c1, c2, c3 = st.columns(3)
                with c1: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#e3e3e3;'>{total}</p><p class='card-meta'>Total Users</p></div>", unsafe_allow_html=True)
                with c2: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#a8c7fa;'>{active}</p><p class='card-meta'>Active Users</p></div>", unsafe_allow_html=True)
                with c3: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#e3e3e3;'>{blocked}</p><p class='card-meta'>Blocked Accounts</p></div>", unsafe_allow_html=True)

                users = get_all_users()
                for uid, uname, uemail, ucreated, ublocked, urole in users:
                    st.markdown(f"<div class='clean-card' style='margin-bottom:10px; padding:15px;'>", unsafe_allow_html=True)
                    colA, colB, colC, colD = st.columns([3, 2, 2, 2])
                    with colA:
                        st.markdown(f"**{uname}** ({uemail})<br><span style='color:#8e918f; font-size:0.8rem;'>Role: {urole.upper()} | Blocked: {'YES' if ublocked else 'NO'}</span>", unsafe_allow_html=True)
                    with colB:
                        if st.button("Promote to Admin" if urole == 'user' else "Demote to User", key=f"role_{uid}", use_container_width=True):
                            toggle_role_user(uemail, urole); st.rerun()
                    with colC:
                        if st.button("Unlock Account" if ublocked else "Lock Account", key=f"lock_{uid}", use_container_width=True):
                            toggle_block_user(uemail, ublocked); st.rerun()
                    with colD:
                        if st.button("Delete User", key=f"del_{uid}", type="primary", use_container_width=True):
                            delete_user(uemail); st.rerun()
                    st.markdown("</div>", unsafe_allow_html=True)

            with tab2:
                st.markdown("### Real-Time Platform Analytics")
                try:
                    df = pd.read_sql("SELECT action_type, details FROM user_activity WHERE action_type NOT IN ('System', 'Security', 'Profile')", conn)
                    if not df.empty:
                        c1, c2 = st.columns(2)
                        with c1:
                            feat_counts = df['action_type'].value_counts().reset_index()
                            feat_counts.columns = ['AI Feature', 'Total Uses']
                            fig1 = px.bar(feat_counts, x='AI Feature', y='Total Uses', title="Popularity of AI Tools", template='plotly_dark', color_discrete_sequence=['#60a5fa'])
                            fig1.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
                            st.plotly_chart(fig1, use_container_width=True)
                        with c2:
                            known_langs = list(LANG_CODES.keys())
                            df['Language'] = df['details'].apply(lambda x: next((l for l in known_langs if l.lower() in str(x).lower()), 'English'))
                            lang_counts = df['Language'].value_counts().reset_index()
                            lang_counts.columns = ['Language', 'Total Output']
                            fig2 = px.pie(lang_counts, names='Language', values='Total Output', title="Regional Language Utilization", template='plotly_dark')
                            fig2.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
                            st.plotly_chart(fig2, use_container_width=True)
                    else:
                        st.info("Not enough user activity data to generate graphs yet.")
                except Exception as e:
                    st.error(f"Could not load analytics: {e}")

            with tab3:
                st.markdown("### User Feedback WordCloud")
                try:
                    fb_df = pd.read_sql("SELECT comments FROM feedback WHERE comments IS NOT NULL AND comments != ''", conn)
                    t_fb_df = pd.read_sql("SELECT comments FROM tool_feedback WHERE comments IS NOT NULL AND comments != ''", conn)
                    all_text = " ".join(fb_df['comments'].tolist() + t_fb_df['comments'].tolist())

                    if all_text.strip():
                        wordcloud = WordCloud(width=800, height=350, background_color='#131314', colormap='Blues', max_words=100).generate(all_text)
                        fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='#131314')
                        ax.imshow(wordcloud, interpolation='bilinear')
                        ax.axis('off')
                        st.pyplot(fig)
                    else:
                        st.info("No written comments submitted by users yet to generate WordCloud.")
                except Exception as e:
                    st.error(f"Error generating WordCloud: {e}")

                st.markdown("---")
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("#### General Platform Feedback")
                    feedbacks = get_all_feedback()
                    if not feedbacks: st.info("No general feedback submitted yet.")
                    for email, rating, comments, ts in feedbacks:
                        st.markdown(f"<div class='clean-card'><p class='card-title'>{email} <span style='float:right; color:#8e918f; font-size:0.8rem;'>{ts}</span></p><p style='color:#a8c7fa; margin:5px 0;'>Rating: {rating}/5</p><p class='card-text'>{comments}</p></div>", unsafe_allow_html=True)
                with c2:
                    st.markdown("#### Individual AI Tool Tuning Data")
                    tool_feeds = get_all_tool_feedback()
                    if not tool_feeds: st.info("No specific AI tool feedback submitted yet.")
                    for email, tool_name, prompt, resp, rating, comments, ts in tool_feeds:
                        st.markdown(f"<div class='clean-card' style='margin-bottom:15px;'><div style='display:flex; justify-content:space-between;'><b style='color:#60a5fa; font-size:0.9rem; text-transform:uppercase;'>{tool_name}</b><span style='color:#8e918f; font-size:0.75rem;'>{ts}</span></div><p style='margin:5px 0; color:#e3e3e3; font-size:0.85rem;'><b>Rating:</b> <span style='color:#81c995;'>{rating}</span></p><p style='font-size:0.85rem; color:#e3e3e3; margin:0;'><b>Notes:</b> {comments if comments else '<i style="color:#8e918f;">No written comment</i>'}</p></div>", unsafe_allow_html=True)

            with tab4:
                st.markdown("### Export System Data")
                st.markdown("<p style='color:#8e918f; font-size:0.9rem; margin-bottom: 20px;'>Download comprehensive database tables as CSV files for local Excel reporting.</p>", unsafe_allow_html=True)
                c1, c2, c3 = st.columns(3)
                try:
                    with c1:
                        users_csv = pd.read_sql("SELECT id, username, email, created_at, role, is_blocked FROM users", conn).to_csv(index=False)
                        st.download_button("Export Full User List", data=users_csv, file_name="users_export.csv", mime="text/csv", use_container_width=True)
                    with c2:
                        activity_csv = pd.read_sql("SELECT * FROM user_activity", conn).to_csv(index=False)
                        st.download_button("Export Activity Logs", data=activity_csv, file_name="activity_logs.csv", mime="text/csv", use_container_width=True)
                    with c3:
                        feed_csv = pd.read_sql("SELECT * FROM tool_feedback", conn).to_csv(index=False)
                        st.download_button("Export AI Tool Feedback", data=feed_csv, file_name="tool_feedback.csv", mime="text/csv", use_container_width=True)
                except Exception as e:
                    st.error("Error connecting to database for export.")

    else:
        if st.session_state.menu_option == "Dashboard": dashboard_page(st.session_state.username, chunks)

        elif st.session_state.menu_option == "Profile":
            st.header("User Profile & Settings")
            tab_id, tab_sec = st.tabs(["Profile Identity", "Security Settings"])

            with tab_id:
                st.markdown("### Profile Identity")
                st.markdown(f"<p style='color:#e3e3e3; font-size:1.4rem; font-weight:600; margin-bottom:0;'>{st.session_state.username}</p>", unsafe_allow_html=True)
                st.markdown(f"<p style='color:#60a5fa; font-size:0.95rem; margin-top:0; margin-bottom:20px;'>{st.session_state.email}</p>", unsafe_allow_html=True)

                user_avatar = get_user_avatar(st.session_state.email)
                if user_avatar:
                    st.markdown(f"<img src='data:image/png;base64,{user_avatar}' style='width:120px; height:120px; border-radius:50%; object-fit:cover; border:3px solid #3b82f6; margin-bottom:20px;'/>", unsafe_allow_html=True)
                else:
                    st.info("No avatar set. Upload one below!")

                new_avatar = st.file_uploader("Upload New Avatar (PNG/JPG - Max 5MB)", type=["png", "jpg", "jpeg"])
                if st.button("Save Avatar", type="primary"):
                    if new_avatar:
                        if new_avatar.size > 5 * 1024 * 1024:
                            st.error("Image file is too large! Please upload an image under 5MB.")
                        else:
                            base64_img = base64.b64encode(new_avatar.read()).decode('utf-8')
                            update_avatar(st.session_state.email, base64_img)
                            log_activity(st.session_state.email, "Profile", "Updated Profile Avatar.")
                            st.success("Avatar updated successfully!")
                            time.sleep(1)
                            st.rerun()
                    else:
                        st.error("Please select an image file first.")

            with tab_sec:
                col1, col2 = st.columns(2)
                with col1:
                    st.markdown("### Update Email Address")
                    with st.form("update_email_form"):
                        new_email = st.text_input("New Email Address", placeholder="new@example.com")
                        curr_pass_email = st.text_input("Current Password", type="password", placeholder="Verify your identity")
                        if st.form_submit_button("Update Email"):
                            if not new_email or not curr_pass_email: st.error("All fields are required.")
                            elif not valid_email(new_email): st.error("Invalid email format.")
                            elif check_user_exists(new_email): st.error("Email already in use.")
                            else:
                                auth_res, _ = authenticate_user(st.session_state.email, curr_pass_email)
                                if auth_res:
                                    update_user_email_db(st.session_state.email, new_email)
                                    log_activity(new_email, "Security", "Email address updated.")
                                    st.success("Email updated successfully! Logging you out...")
                                    time.sleep(1.5)
                                    for key in list(st.session_state.keys()):
                                        del st.session_state[key]
                                    st.rerun()
                                else:
                                    st.error("Incorrect password.")

                with col2:
                    st.markdown("### Change Password")
                    # We remove the st.form here so the UI can update live!
                    old_pass = st.text_input("Current Password", type="password", key="cp_old")
                    new_pass = st.text_input("New Password", type="password", key="cp_new")

                    # Live Password Strength Logic
                    is_len = len(new_pass) >= 6
                    is_upper = bool(re.search(r"[A-Z]", new_pass))
                    is_num = bool(re.search(r"\d", new_pass))
                    is_spec = bool(re.search(r"[^A-Za-z0-9]", new_pass))

                    # Display Live Indicator only when they start typing
                    if new_pass:
                        st.markdown(f"<p style='font-size:0.75rem; margin-top:-10px; margin-bottom:10px; color:#c4c7c5;'>" +
                                    (f"<span style='color:#81c995;'>✅ 6+ chars</span>" if is_len else "❌ 6+ chars") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Uppercase</span>" if is_upper else "❌ Uppercase") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Number</span>" if is_num else "❌ Number") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Special</span>" if is_spec else "❌ Special") +
                                    "</p>", unsafe_allow_html=True)

                    confirm_pass = st.text_input("Confirm New Password", type="password", key="cp_conf")

                    # Replaced form submit button with a standard button
                    if st.button("Update Password", type="primary", use_container_width=True):
                        if not old_pass or not new_pass or not confirm_pass:
                            st.error("All fields required.")
                        elif new_pass != confirm_pass:
                            st.error("New passwords do not match.")
                        elif not (is_len and is_upper and is_num and is_spec):
                            st.error("Password is too weak. Please meet all the security requirements.")
                        else:
                            auth_res, _ = authenticate_user(st.session_state.email, old_pass)
                            if auth_res:
                                if check_password_reused(st.session_state.email, new_pass):
                                    st.error("Cannot reuse a recent password.")
                                else:
                                    update_password(st.session_state.email, new_pass)
                                    st.success("Password updated successfully! Logging you out...")
                                    time.sleep(1.5)
                                    for key in list(st.session_state.keys()): del st.session_state[key]
                                    st.rerun()
                            else:
                                st.error("Incorrect current password.")

        elif st.session_state.menu_option == "History":
            st.header("My Activity History")
            history = get_user_history(st.session_state.email)
            if not history: st.info("No activity recorded yet.")
            else:
                for action, details, prompt, response, ts in history:
                    if prompt or response:
                        with st.expander(f"{action} | {ts}"):
                            st.markdown("**Your Prompt / Input:**"); st.info(prompt)
                            st.markdown("**AI Response:**"); st.success(response)
                    else: st.markdown(f"<div class='clean-card'><p class='card-title'>{action} <span style='float:right; color:#8e918f; font-size:0.8rem; font-weight:400;'>{ts}</span></p><p class='card-text'>{details}</p></div>", unsafe_allow_html=True)

        elif st.session_state.menu_option == "Feedback":
            st.header("Submit General Platform Feedback")
            st.markdown("<p style='color:#c4c7c5;'>Help us improve PolicyNav. Your feedback is sent directly to our administration team.</p>", unsafe_allow_html=True)
            with st.form("feedback_form"):
                rating = st.slider("Rate your experience (1 = Poor, 5 = Excellent)", 1, 5, 5)
                comments = st.text_area("Any suggestions, feature requests, or bugs to report?", height=150)
                if st.form_submit_button("Submit Feedback", type="primary"):
                    if not comments.strip(): st.error("Please write a comment or suggestion before submitting.")
                    else: submit_feedback(st.session_state.email, rating, comments); st.success("Thank you. Your feedback has been recorded.")

        elif st.session_state.menu_option == "Chat":
            st.header("AI Policy Assistant")
            q = st.text_input("Ask a question about government policies:")
            col1, col2, _ = st.columns([2, 3, 5])
            with col1: submit_btn = st.button("Ask AI", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="chat_lang", label_visibility="collapsed")

            if submit_btn and q:
                with st.spinner("Searching..."):
                    eng_q = translate_fast(q, target_lang, "English", t_tokenizer, t_model)
                    context = "\n\n".join(search_policy(eng_q, index, chunks, embed_model))
                    prompt = f"Based on policy context, answer in 2-3 sentences.\n\nContext:\n{context}\n\nQuestion:\n{eng_q}"
                    inputs = s_tokenizer(prompt, return_tensors="pt").to(s_model.device)
                    outputs = s_model.generate(**inputs, max_new_tokens=150)
                    eng_ans = s_tokenizer.decode(outputs[0], skip_special_tokens=True)
                    final_ans = translate_fast(eng_ans, 'English', target_lang, t_tokenizer, t_model)

                    st.session_state.chat_q = q
                    st.session_state.chat_ans = final_ans
                    st.session_state.chat_rated = False
                log_activity(st.session_state.email, "Chat", f"Queried: '{q}'", prompt=q, response=final_ans)

            if st.session_state.get("chat_ans"):
                st.markdown(f"<div class='clean-card'><p class='card-title'>Answer</p><p class='card-text'>{st.session_state.chat_ans}</p></div>", unsafe_allow_html=True)
                if not st.session_state.get("chat_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("chat_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this model by rating the specific response above.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Output Accuracy:", ["Excellent", "Good", "Average", "Poor", "Incorrect / Hallucination"], horizontal=True)
                        t_comment = st.text_input("Optional Notes (e.g., 'Model hallucinated a date' or 'Perfect translation')")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "AI Policy Assistant", st.session_state.chat_q, st.session_state.chat_ans, t_rating, t_comment)
                            st.session_state.chat_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this generation.")

        elif st.session_state.menu_option == "Summarization":
            st.header("AI Policy Summarizer")
            text_to_summarize = st.text_area("Paste policy document text here:", height=250)
            col1, col2, _ = st.columns([2, 3, 5])
            with col1: sum_btn = st.button("Generate Summary", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="sum_lang", label_visibility="collapsed")

            if sum_btn:
                if len(text_to_summarize.strip()) < 50: st.error("Enter at least 50 characters.")
                else:
                    with st.spinner("Processing..."):
                        prompt = f"Summarize in 3-4 sentences.\n\nText: {text_to_summarize[:2000]}"
                        inputs = s_tokenizer(prompt, return_tensors="pt", truncation=True).to(s_model.device)
                        outputs = s_model.generate(**inputs, max_new_tokens=250, min_length=50, num_beams=2, early_stopping=True)
                        eng_summary = s_tokenizer.decode(outputs[0], skip_special_tokens=True)
                        final_summary = translate_fast(eng_summary, "English", target_lang, t_tokenizer, t_model)

                        st.session_state.sum_q = text_to_summarize
                        st.session_state.sum_ans = final_summary
                        st.session_state.sum_rated = False
                    log_activity(st.session_state.email, "Summarization", f"Summary in {target_lang}", prompt=text_to_summarize, response=final_summary)

            if st.session_state.get("sum_ans"):
                st.markdown(f"<div class='clean-card'><p class='card-title'>Summary Result</p><p class='card-text'>{st.session_state.sum_ans}</p></div>", unsafe_allow_html=True)
                if not st.session_state.get("sum_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("sum_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this model by rating the specific response above.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Output Accuracy:", ["Excellent", "Good", "Average", "Poor", "Missed Context / Too Long"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "AI Policy Summarizer", st.session_state.sum_q, st.session_state.sum_ans, t_rating, t_comment)
                            st.session_state.sum_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this generation.")

        elif st.session_state.menu_option == "Graph":
            st.header("Entity Knowledge Graph")
            if st.button("Generate Graph", type="primary"):
                log_activity(st.session_state.email, "Knowledge Graph", "Rendered graph")
                with st.spinner("Extracting..."): generate_knowledge_graph(chunks[:15], nlp); components.html(open("policy_graph.html", 'r', encoding='utf-8').read(), height=450)

        elif st.session_state.menu_option == "Web":
            st.header("Global Web Search")
            query = st.text_input("Enter research query:")

            if st.button("Search Web", type="primary") and query:
                with st.spinner("Retrieving..."):
                    results = global_web_research(query)
                    formatted_results = ""
                    for r in results:
                        st.markdown(f"<div class='clean-card'><p class='card-title'>{r['title']}</p><p class='card-text'>{r['body']}</p><a href='{r['href']}' style='color:#a8c7fa; font-size:0.85rem; text-decoration:none;'>Read source document</a></div>", unsafe_allow_html=True)
                        formatted_results += f"**{r['title']}**\n{r['body']}\n\n"

                    st.session_state.web_q = query
                    st.session_state.web_ans = formatted_results
                    st.session_state.web_rated = False
                log_activity(st.session_state.email, "Web Research", f"Researched: '{query}'", prompt=query, response=formatted_results)

            if st.session_state.get("web_ans"):
                if not st.session_state.get("web_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("web_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this search tool by rating the results.</p>", unsafe_allow_html=True)
                        t_rating = radio("Search Relevance:", ["Excellent", "Good", "Average", "Poor", "Irrelevant Results"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Global Web Search", st.session_state.web_q, st.session_state.web_ans, t_rating, t_comment)
                            st.session_state.web_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this search.")

        elif st.session_state.menu_option == "Readability":
            st.header("Readability Analyzer")
            tab1, tab2 = st.tabs(["Paste Text", "Upload File"]); text_input = ""
            with tab1:
                raw_text = st.text_area("Enter text below", height=180, key="read_txt")
                if raw_text: text_input = raw_text
            with tab2:
                uploaded_file = st.file_uploader("Upload file", type=["txt", "pdf"])
                if uploaded_file:
                    if uploaded_file.type == "application/pdf": text_input = "".join([page.extract_text() + "\n" for page in PyPDF2.PdfReader(uploaded_file).pages])
                    else: text_input = uploaded_file.read().decode("utf-8")

            if st.button("Analyze Text", type="primary"):
                if len(text_input.strip()) < 50: st.error("Text too short.")
                else:
                    with st.spinner("Calculating..."):
                        scores = readability.ReadabilityAnalyzer(text_input).get_all_metrics()

                        st.session_state.read_q = text_input[:200] + "..."
                        st.session_state.read_ans = f"Flesch Ease: {scores['Flesch Reading Ease']:.1f} | Grade: {scores['Flesch-Kincaid Grade']:.1f}"
                        st.session_state.read_scores = scores
                        st.session_state.read_rated = False
                    log_activity(st.session_state.email, "Readability", "Analyzed metrics", prompt=st.session_state.read_q, response=st.session_state.read_ans)

            if st.session_state.get("read_scores"):
                st.markdown("<br>", unsafe_allow_html=True); c1, c2, c3 = st.columns(3)
                scores = st.session_state.read_scores
                with c1: st.markdown(metric_card("Flesch Ease", "0-100 score", scores["Flesch Reading Ease"], scores["Flesch Reading Ease"], "#a8c7fa"), unsafe_allow_html=True)
                with c2: st.markdown(metric_card("Flesch-Kincaid", "Grade level", scores["Flesch-Kincaid Grade"], (scores["Flesch-Kincaid Grade"] / 20) * 100, "#81c995"), unsafe_allow_html=True)
                with c3: st.markdown(metric_card("SMOG Index", "Education level", scores["SMOG Index"], (scores["SMOG Index"] / 20) * 100, "#f28b82"), unsafe_allow_html=True)

                if not st.session_state.get("read_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("read_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune the readability analyzer.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Accuracy of Math/Analysis:", ["Excellent", "Good", "Average", "Poor", "Failed to Parse Text"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Readability Analyzer", st.session_state.read_q, st.session_state.read_ans, t_rating, t_comment)
                            st.session_state.read_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this analysis.")

        elif st.session_state.menu_option == "Translation":
            st.header("Language Translator")
            source_lang = st.selectbox("Detect Source Language:", list(LANG_CODES.keys()), index=0)
            text_to_translate = st.text_area("Paste text to translate:", height=200)
            col1, col2, _ = st.columns([2, 3, 5])
            with col1: trans_btn = st.button("Translate Text", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="trans_lang", label_visibility="collapsed")

            if trans_btn:
                if text_to_translate.strip() == "": st.error("Please paste text.")
                else:
                    with st.spinner("Translating..."):
                        result = translate_fast(text_to_translate, source_lang, target_lang, t_tokenizer, t_model)

                        st.session_state.trans_q = text_to_translate
                        st.session_state.trans_ans = result
                        st.session_state.trans_rated = False
                    log_activity(st.session_state.email, "Translation", f"To {target_lang}", prompt=text_to_translate, response=result)

            if st.session_state.get("trans_ans"):
                st.markdown(f"<div class='clean-card'><p class='card-title'>Translation</p><p class='card-text'>{st.session_state.trans_ans}</p></div>", unsafe_allow_html=True)

                if not st.session_state.get("trans_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("trans_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this translation model.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Translation Accuracy:", ["Excellent", "Good", "Average", "Poor Grammar", "Incorrect Language / Output"], horizontal=True)
                        t_comment = st.text_input("Optional Notes (e.g., 'Grammar was messy')")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Language Translator", st.session_state.trans_q, st.session_state.trans_ans, t_rating, t_comment)
                            st.session_state.trans_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this translation.")

else:
    if st.session_state.page == "signup": signup()
    elif st.session_state.page == "admin_login": admin_login()
    elif st.session_state.page == "forgot": forgot_password()
    else: login()

Overwriting app.py


In [15]:
!pip install pyngrok streamlit -q

In [23]:
import os
import time
import subprocess
import warnings
from pyngrok import ngrok
from google.colab import userdata

# 1. Grab secrets from Colab and inject them into the system environment
os.environ['EMAIL_ID'] = userdata.get('EMAIL_ID')
os.environ['EMAIL_APP_PASSWORD'] = userdata.get('EMAIL_APP_PASSWORD')

# 2. Mute Colab warnings
warnings.filterwarnings("ignore")

# 3. Kill old sessions cleanly
os.system("pkill -f streamlit")
ngrok.kill()

# 4. Connect Ngrok securely
# We now grab the token directly from your Secrets tab so it stays hidden!
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8501).public_url

print("✅ ALL SYSTEMS GO!")
print("👇 Click below to open your Secure Integrated Dashboard 👇")
print(f"🔗 {public_url}")
print("--------------------------------------------------")

# 5. Start Streamlit (It will now inherit the email secrets!)
subprocess.Popen(['nohup', 'streamlit', 'run', 'app.py', '--server.port', '8501'])

✅ ALL SYSTEMS GO!
👇 Click below to open your Secure Integrated Dashboard 👇
🔗 https://unannealed-nondiastatic-barbara.ngrok-free.dev
--------------------------------------------------


<Popen: returncode: None args: ['nohup', 'streamlit', 'run', 'app.py', '--se...>

In [21]:
!pkill -f streamlit
!pkill -f ngrok